In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class AlcoholToAlkene(MorphingOperator):
    def __init__(self):
        super(AlcoholToAlkene, self).__init__()
        self._name = "Saytzeff Dehydration"
        self._target_groups = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")

    def setOriginal(self, mol):
        super(AlcoholToAlkene, self).setOriginal(mol)
        self._target_groups = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_groups.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_groups: return MolpherMol(other=rdkit_mol)
            
        oh_idx, alpha_idx = random.choice(self._target_groups)
        alpha_atom = rdkit_mol.GetAtomWithIdx(alpha_idx)
        
        beta_carbons = [a for a in alpha_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetHybridization() == Chem.HybridizationType.SP3]
        if not beta_carbons: return MolpherMol(other=rdkit_mol)

        beta_carbons.sort(key=lambda x: x.GetTotalNumHs())
        beta_idx = beta_carbons[0].GetIdx()
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond_ab = rw_mol.GetBondBetweenAtoms(alpha_idx, beta_idx)
            if bond_ab:
                bond_ab.SetBondType(Chem.BondType.DOUBLE)
                
            bond_oh = rw_mol.GetBondBetweenAtoms(oh_idx, alpha_idx)
            if bond_oh:
                rw_mol.RemoveBond(oh_idx, alpha_idx)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [alpha_idx, beta_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags = Chem.GetMolFrags(new_mol, asMols=True)
            if frags:

                frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
                final_mol = frags[0]
                
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name

op_dehydration = AlcoholToAlkene()

dehydration_tests = {
    "1. 2-Βουτανόλη (Αλκοόλη -> Saytzeff)": "CCC(O)C",
    "2. Φαινόλη (Αρωματικό OH -> Πρέπει να αγνοηθεί)": "Oc1ccccc1",
    "3. Οξικό οξύ (Καρβοξύλιο -> Πρέπει να αγνοηθεί)": "CC(=O)O"
}

print("\n=== STARTING SAYTZEFF DEHYDRATION TESTING ===")
for name, smiles in dehydration_tests.items():
    mol = MolpherMol(smiles)
    op_dehydration.setOriginal(mol)
    product = op_dehydration.morph()
    print(f"\n{name}\n  SOURCE: {mol.getSMILES()}\n  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n==============================================")


=== STARTING SAYTZEFF DEHYDRATION TESTING ===

1. 2-Βουτανόλη (Αλκοόλη -> Saytzeff)
  SOURCE: CCC(C)O
  TARGET: CC=CC

2. Φαινόλη (Αρωματικό OH -> Πρέπει να αγνοηθεί)
  SOURCE: OC1=CC=CC=C1
  TARGET: No change (Safe)

3. Οξικό οξύ (Καρβοξύλιο -> Πρέπει να αγνοηθεί)
  SOURCE: CC(=O)O
  TARGET: No change (Safe)



In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class AlcoholToAlkene(MorphingOperator):
    def __init__(self):
        super(AlcoholToAlkene, self).__init__()
        self._name = "Saytzeff Dehydration"
        self._target_groups = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")

    def setOriginal(self, mol):
        super(AlcoholToAlkene, self).setOriginal(mol)
        self._target_groups = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_groups.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_groups: return MolpherMol(other=rdkit_mol)
            
        oh_idx, alpha_idx = random.choice(self._target_groups)
        alpha_atom = rdkit_mol.GetAtomWithIdx(alpha_idx)
        
        beta_carbons = [a for a in alpha_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetHybridization() == Chem.HybridizationType.SP3]
        if not beta_carbons: return MolpherMol(other=rdkit_mol)

        beta_carbons.sort(key=lambda x: x.GetTotalNumHs())
        beta_idx = beta_carbons[0].GetIdx()
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond_ab = rw_mol.GetBondBetweenAtoms(alpha_idx, beta_idx)
            if bond_ab:
                bond_ab.SetBondType(Chem.BondType.DOUBLE)
                
            bond_oh = rw_mol.GetBondBetweenAtoms(oh_idx, alpha_idx)
            if bond_oh:
                rw_mol.RemoveBond(oh_idx, alpha_idx)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [alpha_idx, beta_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags = Chem.GetMolFrags(new_mol, asMols=True)
            if frags:

                frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
                final_mol = frags[0]
                
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name

op_dehydration = AlcoholToAlkene()
start_mol = MolpherMol("CCC(C)O")          
target_mol = MolpherMol("CC=CC")   
tree = ETree.create(source=start_mol, target=target_mol)  
tree.morphing_operators = (op_dehydration,)

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()
max_generations = 5

print("=== STARTING SAYTZEFF DEHYDRATION SEARCH ===")
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print('Generation #', tree.generation_count, sep='')
    print('Molecules in tree:', tree.mol_count)
    if closest_info.closest_mol:
        print('Closest molecule to target: {0} (Tanimoto distance: {1})'.format(
            closest_info.closest_mol.getSMILES(), closest_info.closest_distance))
    print("-" * 40)

if tree.path_found:
    print("SUCCESS: Το Molpher βρήκε το μονοπάτι αφυδάτωσης!")
else:
    print("FAILED: Δεν βρέθηκε το μονοπάτι.")
print("=========================================")

=== STARTING SAYTZEFF DEHYDRATION SEARCH ===
Generation #1
Molecules in tree: 2
Closest molecule to target: CC=CC (Tanimoto distance: 0.0)
----------------------------------------
SUCCESS: Το Molpher βρήκε το μονοπάτι αφυδάτωσης!
